# 17 — Τεκμήρια πριν από την ταξινόμηση ζητημάτων

Σύγκριση **F: διορθωμένη ανάλυση δύο σταδίων** με **G: χωριστά τεκμήρια και UNCERTAIN**, στο ίδιο Qwen 1.5B και τις ίδιες 24 επινοημένες κριτικές ανάπτυξης.

Και τα δύο έχουν έως δύο κλήσεις των 400 tokens. Η πραγματική χρήση κλήσεων και ο χρόνος μπορεί να διαφέρουν. Το G διατηρεί χωριστά αναφορά προβλήματος, υποθετική διατύπωση και ρητή επίλυση. Οι αμφίβολες περιπτώσεις παραμένουν ορατές. Το τμήμα προκύπτει από παραμετροποιήσιμο πίνακα κατηγοριών.

**GPU Colab → Run all. Δεν χρειάζεται ZIP ή API key.** Οι αποθηκευμένες προβλέψεις DistilBERT/Jev του notebook 16 διαβάζονται από το αρχειοθετημένο run στο repo και ελέγχονται με SHA-256. Νέες κλήσεις Jev: 0.

Δεν υπάρχει αποτέλεσμα ποιότητας ακόμη. Συμπλήρωσε τα `human_review.csv` και `coverage_review.csv` μετά την εκτέλεση. Δεν γίνεται εκπαίδευση, επιλογή νικητή, τελικός ανεξάρτητος έλεγχος ή δημοσίευση στο Space.

## 1. Καρφωμένος κώδικας και προηγούμενο run

In [ ]:
from pathlib import Path

SOURCE_COMMIT = "22348e703daff90e9e6848017fdc1159f898832e"
SCRIPT_SHA256 = "4f3dab80b7df527d9192de03a67504cf8569dc7ca11b53347cf11c417c45bd66"
REVIEWS_SHA256 = "69718a4170ca151fe880455f17b243b79bc7eaa04ce4a9eea8048ef8f83ec6f8"
REFERENCE_RUN_SHA256 = "78904f6b466fa5e6ea03a1d0de5db330059194e82f5b2e5a4aa2fea8aac66534"
REFERENCE_UPSTREAM_SHA256 = "3d57b39b8350383beedb4f708eaf8b78b91196a16a2757fb52b923eefa728235"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
REPO_DIR = Path("/content/hotel-review-evidence-v5")
REFERENCE_RELATIVE = "docs/experiments/triage_generator/runs/20261003T145901Z_a29e8d68"
SECRET_NAME = "OPENROUTER_API_KEY"
EXPECTED_CANDIDATES = ("F", "G")
REFERENCE_READY = SETUP_READY = GPU_READY = False
MODULE_HASHES = {'src/reviewnlp/triage/qwen_generator.py': 'e6743eaa68bd9b4cb8596b72681c8fb8c43945abe5ba3a9f97812f1c940d839d', 'src/reviewnlp/triage/generator_experiment.py': '33e71055e9d6f168ca6d4e2422ca27da64988990e047a951e5ca7dc8aef0bc06', 'src/reviewnlp/triage/staged_generator.py': '5393fb6891e72545300be4926287c131468163b9fc3ade3e951fc60e77a51517', 'src/reviewnlp/triage/evidence_generator.py': '959f78e85963385e30ec275451ee2e5f92eef2cbe39dc67b355678a939b61bc3', 'src/reviewnlp/triage/workflow_review.py': '4777088a4d0bf6e118eeab12ad7d8b906c7097d79963fb97040afd0629c442c9', 'src/reviewnlp/triage/pipeline.py': 'de8883a7d806489246bac0b4222775520c45902670527f7aded43669cc62fcef'}
print("Κώδικας:", SOURCE_COMMIT)
print("24 κριτικές ανάπτυξης, F/G, ίδιο Qwen 1.5B. Νέες κλήσεις Jev: 0.")


### 2. Προετοιμασία του καρφιτσωμένου κώδικα

Χρησιμοποιεί το GPU-enabled torch του Colab. Αφαιρεί ασύμβατα πακέτα που δεν χρειάζεται η σύγκριση και εγκαθιστά τις ίδιες εκδόσεις με τα προηγούμενα runs.

In [ ]:
import hashlib
import os
import importlib.metadata as metadata
import json
import subprocess
import sys

SETUP_READY = False
GPU_READY = False

setup_env = {name: value for name, value in os.environ.items() if name != SECRET_NAME}

def setup_command(arguments, *, cwd=None, capture=False):
    completed = subprocess.run(
        arguments, cwd=cwd, env=setup_env, check=True, text=True,
        stdout=subprocess.PIPE if capture else None,
    )
    return completed.stdout.strip() if capture else None

if not REPO_DIR.exists():
    setup_command([
        "git", "clone", "--quiet",
        "https://github.com/krimits/hotel-review-nlp.git", str(REPO_DIR),
    ])
    setup_command(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], cwd=REPO_DIR)
current_commit = setup_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True)
if current_commit != SOURCE_COMMIT:
    raise RuntimeError("Το checkout έχει διαφορετικό commit. Χρησιμοποίησε νέα συνεδρία Colab.")
if setup_command(["git", "status", "--porcelain", "--untracked-files=no"],
                 cwd=REPO_DIR, capture=True):
    raise RuntimeError("Υπάρχουν αλλαγές στα αρχεία του checkout. Δεν γίνεται αυτόματο reset.")

def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

SCRIPT_PATH = REPO_DIR / "scripts/compare_triage_generators.py"
REVIEWS_PATH = REPO_DIR / "docs/experiments/triage_generator/dev.json"
if file_sha256(SCRIPT_PATH) != SCRIPT_SHA256 or file_sha256(REVIEWS_PATH) != REVIEWS_SHA256:
    raise RuntimeError("Τα fingerprints του script ή των κριτικών δεν συμφωνούν.")
for relative_path, expected in MODULE_HASHES.items():
    if file_sha256(REPO_DIR / relative_path) != expected:
        raise RuntimeError("Διαφέρει το module: " + relative_path)

fixture_reviews = json.loads(REVIEWS_PATH.read_text(encoding="utf-8"))["reviews"]
EXPECTED_IDS = {review["id"] for review in fixture_reviews}
if len(fixture_reviews) != 24 or len(EXPECTED_IDS) != 24:
    raise RuntimeError("Το fixture πρέπει να έχει 24 διαφορετικά IDs.")

# Reuse Colab's GPU-enabled torch. Do not download or upgrade torch here.
try:
    metadata.version("torch")
except metadata.PackageNotFoundError:
    raise RuntimeError("Δεν υπάρχει PyTorch. Άνοιξε το notebook σε συνεδρία Colab με GPU.") from None

unused_packages = []
for package in ("gradio", "gradio-client", "diffusers"):
    try:
        metadata.version(package)
        unused_packages.append(package)
    except metadata.PackageNotFoundError:
        pass
pip = [sys.executable, "-m", "pip", "--disable-pip-version-check"]
if unused_packages:
    setup_command(pip + ["uninstall", "--quiet", "-y", *unused_packages])
requirements = [
    "transformers==4.56.2", "huggingface-hub==0.36.2", "accelerate==1.10.1",
    "fastapi>=0.110", "httpx>=0.27",
]
core_requirements = {
    "numpy": "numpy>=1.26", "pandas": "pandas>=2.1", "pyarrow": "pyarrow>=14.0",
    "joblib": "joblib>=1.3", "scikit-learn": "scikit-learn>=1.4", "scipy": "scipy>=1.11",
    "pyyaml": "pyyaml>=6.0", "tqdm": "tqdm>=4.66", "matplotlib": "matplotlib>=3.8",
    "seaborn": "seaborn>=0.13",
}
for package, requirement in core_requirements.items():
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        requirements.append(requirement)
setup_command(pip + ["install", "--quiet", *requirements])
setup_command(pip + ["install", "--quiet", "--no-deps", "-e", str(REPO_DIR)])
SETUP_READY = True
print("Checkout και fingerprints: OK.")

### 3. Έλεγχος της αρχειοθετημένης αναφοράς

Καμία μεταφόρτωση από τον χρήστη. Τα δεδομένα αναφοράς είναι συνθετικά, έχουν ήδη καταγραφεί στο repo και επαληθεύονται πριν την παραγωγή.

In [ ]:
import json

REFERENCE_READY = False
if not SETUP_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα την προετοιμασία.")
REFERENCE_DIR = REPO_DIR / REFERENCE_RELATIVE
if (file_sha256(REFERENCE_DIR / "run.json") != REFERENCE_RUN_SHA256
        or file_sha256(REFERENCE_DIR / "upstream.json") != REFERENCE_UPSTREAM_SHA256):
    raise RuntimeError("Διαφέρει η αρχειοθετημένη αναφορά του notebook 16.")
reference_info = json.loads((REFERENCE_DIR / "run.json").read_text())
if (reference_info.get("split") != "dev" or reference_info.get("execution_complete") is not True
        or reference_info.get("dataset_sha256") != REVIEWS_SHA256
        or reference_info["candidates"]["F"]["revision"] != MODEL_REVISION):
    raise RuntimeError("Η αναφορά δεν έχει τα αναμενόμενα δεδομένα ή βάρη.")
REFERENCE_READY = True
print("Αρχειοθετημένο run και upstream: OK. Δεν χρειάζεται API key.")


## 4. GPU και έλεγχος imports σε νέο Python process

In [ ]:
import importlib.metadata as metadata
import textwrap

GPU_READY = False
if not SETUP_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα το κελί προετοιμασίας.")
environment_code = """
import json
import torch
import transformers
import huggingface_hub
from reviewnlp.triage.pipeline import TriagePipeline
from reviewnlp.triage.qwen_generator import QwenActionGenerator

if not torch.cuda.is_available():
    raise RuntimeError("Δεν υπάρχει GPU. Colab: Runtime -> Change runtime type -> GPU.")
assert transformers.__version__ == "4.56.2", "Διαφορετική έκδοση transformers"
assert huggingface_hub.__version__ == "0.36.2", "Διαφορετική έκδοση huggingface-hub"
assert metadata.version("accelerate") == "1.10.1", "Διαφορετική έκδοση accelerate"
print(json.dumps({
    "python": __import__("platform").python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "accelerate": metadata.version("accelerate"),
    "gpu": torch.cuda.get_device_name(0),
}))
"""

environment_json = setup_command(
    [sys.executable, "-c", textwrap.dedent(environment_code)], cwd=REPO_DIR, capture=True
)
environment_info = json.loads(environment_json)
for name, value in environment_info.items():
    print(f"{name}: {value}")
GPU_READY = True

## 5. Σύγκριση C έναντι F μόνο στα development cases

Το script επαληθεύει ξανά τη συμβατότητα του προηγούμενου run. Επαναχρησιμοποιεί τα
upstream bytes και το προηγούμενο model revision. Δεν κάνει κλήσεις στο Jev ούτε
φορτώνει DistilBERT. Φόρτωση/warmup και χρόνος κάθε κριτικής καταγράφονται χωριστά.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

if not REFERENCE_READY:
    raise RuntimeError("Πρώτα έλεγξε την αρχειοθετημένη αναφορά.")
if not GPU_READY:
    raise RuntimeError("Πρώτα ολοκλήρωσε τον έλεγχο GPU.")
if file_sha256(SCRIPT_PATH) != SCRIPT_SHA256 or file_sha256(REVIEWS_PATH) != REVIEWS_SHA256:
    raise RuntimeError("Το script ή τα δεδομένα άλλαξαν.")
for relative_path, expected in MODULE_HASHES.items():
    if file_sha256(REPO_DIR / relative_path) != expected:
        raise RuntimeError("Το module άλλαξε: " + relative_path)
run_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RUN_DIR = REPO_DIR / "runs" / ("generator_evidence_dev_" + run_tag)
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR.parent / (RUN_DIR.name + "_execution.txt")
command = [sys.executable, "-u", "scripts/compare_triage_generators.py", "dev",
           "--candidates", "F", "G", "--reference-run", str(REFERENCE_DIR),
           "--output", str(RUN_DIR)]
process_env = {name: value for name, value in os.environ.items() if name != SECRET_NAME}
process_env["TOKENIZERS_PARALLELISM"] = "false"
print("Run:", RUN_DIR.name)
with LOG_PATH.open("w", encoding="utf-8") as log:
    with subprocess.Popen(command, cwd=REPO_DIR, env=process_env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            if os.environ.get(SECRET_NAME):
                line = line.replace(os.environ[SECRET_NAME], "[REDACTED]")
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
del process_env
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH.replace(RUN_DIR / "execution.txt")
print("Exit:", exit_code)
print("Συνέχισε για έλεγχο και κατέβασμα της αναφοράς, ακόμη και αν υπήρξε σφάλμα.")

## 6. Πληρότητα και σύνοψη

Ελέγχει 24 IDs για κάθε workflow, ίδιες upstream προβλέψεις, ίδιο Qwen revision και
μηδενικές νέες κλήσεις API. Ξεχωρίζει το πλήρες JSON χωρίς διπλά κλειδιά από την
αποδοχή αποσπασμάτων του parser. Κανένας από αυτούς τους ελέγχους δεν αποδεικνύει
ότι μια πρόταση είναι σωστή ή χρήσιμη.

In [ ]:
from collections import Counter
from IPython.display import Markdown, display

issues = []
required = ("run.json", "results.jsonl", "upstream.json", "human_review.csv", "annotation_key.json", "summary.md", "reference_run.json", "coverage_review.csv")
missing = [name for name in required if not (RUN_DIR / name).is_file()]
if exit_code != 0:
    issues.append(f"Exit code {exit_code}.")
if missing:
    issues.append("Λείπουν αρχεία: " + ", ".join(missing))
else:
    info = json.loads((RUN_DIR / "run.json").read_text())
    records = [json.loads(line) for line in (RUN_DIR / "results.jsonl").read_text().splitlines()]
    upstream = json.loads((RUN_DIR / "upstream.json").read_text())
    if not info["execution_complete"] or info["quality_evaluated"]:
        issues.append("Λανθασμένη κατάσταση εκτέλεσης ή ανθρώπινης αξιολόγησης.")
    if info["source_commit"] != SOURCE_COMMIT or info["dataset_sha256"] != REVIEWS_SHA256:
        issues.append("Διαφέρουν ο κώδικας ή τα δεδομένα.")
    if info["split"] != "dev" or set(info["candidates"]) != set(EXPECTED_CANDIDATES):
        issues.append("Δεν εκτελέστηκε η σωστή σύγκριση ανάπτυξης.")
    if not info.get("upstream_reused") or info["api_cost"]["attempts"] != 0:
        issues.append("Έγιναν νέες κλήσεις API ή δεν επαναχρησιμοποιήθηκε το upstream.")
    if (info.get("reference_run", {}).get("run_sha256") != REFERENCE_RUN_SHA256
            or file_sha256(RUN_DIR / "upstream.json") != REFERENCE_UPSTREAM_SHA256):
        issues.append("Διαφέρει η αναφορά ή το upstream από το προηγούμενο run.")
    if any(config.get("revision") != MODEL_REVISION for config in info["candidates"].values()):
        issues.append("Δεν χρησιμοποιήθηκαν τα προηγούμενα βάρη του Qwen.")
    if Counter(row["id"] for row in upstream["records"]) != Counter(EXPECTED_IDS):
        issues.append("Το upstream δεν καλύπτει τα 24 IDs μία φορά.")
    if any(row["complaints"]["status"] != "ok" for row in upstream["records"]):
        issues.append("Αποτυχία Jev.")
    stage_hashes = {}
    for candidate in EXPECTED_CANDIDATES:
        subset = [row for row in records if row["candidate"] == candidate]
        if Counter(row["id"] for row in subset) != Counter(EXPECTED_IDS):
            issues.append(candidate + ": λείπουν ή επαναλαμβάνονται IDs.")
        if any(row["error"] for row in subset):
            issues.append(candidate + ": σφάλμα παραγωγής.")
        for row in subset:
            previous = stage_hashes.setdefault(row["id"], row["upstream_sha256"])
            if previous != row["upstream_sha256"]:
                issues.append("Δεν χρησιμοποιήθηκαν ίδιες προβλέψεις upstream.")
    expected_budgets = {"G": {"max_new_tokens_per_call": 400, "max_calls_per_review": 2},
                        "F": {"max_new_tokens_per_call": 400, "max_calls_per_review": 2}}
    if info.get("generation_budget") != expected_budgets:
        issues.append("Δεν καταγράφηκαν σωστά τα ίσα μέγιστα budgets.")
    for candidate in EXPECTED_CANDIDATES:
        staged_records = [row for row in records if row["candidate"] == candidate]
        if any(not row.get("stages") or row["stages"][0]["stage"] != "issues" for row in staged_records):
            issues.append(candidate + ": λείπει η καταγραφή εξαγωγής ζητημάτων.")
        measure_calls = sum(stage["stage"] == "measures" for row in staged_records for stage in row.get("stages", []))
        if not measure_calls:
            issues.append(candidate + ": δεν εκτελέστηκε το δεύτερο στάδιο· δες τα raw issues.")
        print(candidate, "κλήσεις παραγωγής μέτρων:", measure_calls)
    print("Η ολοκλήρωση δεν αποδεικνύει χρήσιμα μέτρα. Ελέγξτε τα raw stages και τις εξαιρέσεις.")
    for name, expected in info["files"].items():
        if file_sha256(RUN_DIR / name) != expected:
            issues.append("Διαφέρει το fingerprint του " + name)
    display(Markdown((RUN_DIR / "summary.md").read_text()))
    display(Markdown("**Δεν επιλέχθηκε νικητής.** Συμπλήρωσε τα human_review.csv και coverage_review.csv σύμφωνα με το πρωτόκολλο."))
checks = {"source_commit": SOURCE_COMMIT, "process_exit_code": exit_code,
          "issues": list(dict.fromkeys(issues)), "environment": environment_info,
          "human_quality_evaluated": False}
(RUN_DIR / "notebook_checks.json").write_text(json.dumps(checks, ensure_ascii=False, indent=2) + "\n")
print("Έλεγχοι εκτέλεσης:", "OK" if not issues else "; ".join(issues))

In [ ]:
# Descriptive diagnostics only; these counts are not human quality scores.
import matplotlib.pyplot as plt

if not missing:
    names = list(EXPECTED_CANDIDATES)
    counts = info["summary"]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for axis, key, title in zip(axes,
        ("reviews_with_accepted_actions", "workflow_failures"),
        ("Κριτικές με δομικά αποδεκτές προτάσεις", "Αποτυχίες workflow")):
        values = [counts[name][key] for name in names]
        axis.bar(names, values, color=["#355C7D", "#2A9D8F"])
        axis.set_ylim(0, 24)
        axis.set_title(title, fontsize=11)
        axis.set_ylabel("Πλήθος από 24 συνθετικές κριτικές")
        for index, value in enumerate(values):
            axis.text(index, value + 0.4, str(value), ha="center")
    fig.suptitle("Έλεγχοι δομής — η ανθρώπινη αξιολόγηση εκκρεμεί", fontsize=12)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "structure.png", dpi=140, bbox_inches="tight")
    plt.show()


## 7. Κατέβασμα

Το ZIP περιλαμβάνει ακατέργαστες απαντήσεις, αποδεκτά μέτρα, revisions, fingerprints,
χρόνους, διαθέσιμη χρήση/κόστος Jev, log και κενό φύλλο ανθρώπινης αξιολόγησης.
Αποθηκεύεται και σε αποτυχημένο run για διάγνωση. Δώσε στον αξιολογητή μόνο το
human_review.csv· κράτησε χωριστά το annotation_key.json με τις ταυτότητες των παραλλαγών.

In [ ]:
import zipfile
from google.colab import files

export_paths = sorted(path for path in RUN_DIR.iterdir() if path.is_file())
_secret_for_check = os.environ.get(SECRET_NAME, "")
if _secret_for_check and any(_secret_for_check.encode() in path.read_bytes() for path in export_paths):
    del _secret_for_check
    raise RuntimeError("Δεν εξάγεται αναφορά που περιέχει Secret.")
del _secret_for_check
ZIP_PATH = RUN_DIR.parent / (RUN_DIR.name + ".zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in export_paths:
        archive.write(path, arcname=path.name)
print("ZIP:", ZIP_PATH.name)
files.download(str(ZIP_PATH))
if issues:
    raise RuntimeError("Το ZIP κατέβηκε, αλλά η εκτέλεση έχει προβλήματα. Δες το execution.txt.")

### Επόμενα βήματα

Το ZIP περιέχει τα πραγματικά model outputs, τα fingerprints, τους χρόνους και δύο φύλλα προς ανθρώπινη αξιολόγηση. Δεν επιλέχθηκε νικητής.

- `human_review.csv`: χρησιμότητα, τεκμηρίωση, σωστό τμήμα, απουσία επινοημένων γεγονότων.
- `coverage_review.csv`: πραγματικά αναφερθέντα προβλήματα, χαμένα προβλήματα, λανθασμένες εξαιρέσεις, ζητήματα χωρίς κατάλληλο μέτρο, επινοήσεις και ποιότητα ενεργειών. Ο ανθρώπινος αριθμός πραγματικών προβλημάτων πρέπει να είναι ίδιος για το ίδιο review και στα δύο σκέλη.
- Στείλε το ZIP και τα συμπληρωμένα φύλλα για έλεγχο. Οι δομικές μετρήσεις και ο αριθμός προτάσεων δεν αποδεικνύουν ποιότητα. Ο τελικός ανεξάρτητος έλεγχος παραμένει κλειστός.

Πρωτόκολλο: `docs/experiments/triage_generator/evidence_protocol.md`. Το demo προετοιμάζεται χωριστά και δεν αλλάζει η δημοσιευμένη εφαρμογή από αυτό το notebook.
